In [1]:
import lightgbm as lgb
print(lgb.__version__)

4.7.0


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/karhut_ml_dataset.csv")

df["date"] = pd.to_datetime(df["date"])

print("Shape:", df.shape)
print("Date:", df["date"].min(), "to", df["date"].max())

Shape: (34272, 39)
Date: 2026-05-08 00:00:00 to 2026-09-10 00:00:00


In [3]:
features = [
    # Fire History
    "fire_count_lag1",
    "fire_count_lag3",
    "fire_count_lag7",
    "frp_sum_lag1",
    "frp_sum_lag3",
    "frp_sum_lag7",
    "fire_count_roll3",
    "fire_count_roll7",
    "frp_sum_roll3",
    "frp_sum_roll7",

    # Weather
    "temperature_mean",
    "temperature_max",
    "relative_humidity_mean",
    "precipitation_sum",
    "boundary_layer_height_mean",
    "vapour_pressure_deficit_mean",
    "wind_speed_mean",
    "wind_speed_max",
    "wind_u_mean",
    "wind_v_mean"
]

target = "target_fire_active_t1"

X = df[features]
y = df[target].astype(int)

print("Number of features:", len(features))
print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts())

Number of features: 20
X shape: (34272, 20)
y distribution:
target_fire_active_t1
0    27990
1     6282
Name: count, dtype: int64


In [4]:
train_mask = df["date"] < "2026-08-01"

val_mask = (
    (df["date"] >= "2026-08-01") &
    (df["date"] < "2026-09-01")
)

test_mask = df["date"] >= "2026-09-01"

X_train = X[train_mask]
y_train = y[train_mask]

X_val = X[val_mask]
y_val = y[val_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print("TRAIN")
print(X_train.shape, "Positive rate:", y_train.mean())

print("\nVALIDATION")
print(X_val.shape, "Positive rate:", y_val.mean())

print("\nTEST")
print(X_test.shape, "Positive rate:", y_test.mean())

TRAIN
(23120, 20) Positive rate: 0.11042387543252595

VALIDATION
(8432, 20) Positive rate: 0.3391840607210626

TEST
(2720, 20) Positive rate: 0.31948529411764703


In [5]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

print("Negative:", neg)
print("Positive:", pos)
print("Scale Pos Weight:", scale_pos_weight)

Negative: 20567
Positive: 2553
Scale Pos Weight: 8.056012534273403


In [6]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgbm.fit(
    X_train,
    y_train
)

print("LightGBM training selesai.")

LightGBM training selesai.


In [7]:
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)

val_proba = lgbm.predict_proba(X_val)[:, 1]

val_pred = (val_proba >= 0.5).astype(int)

pr_auc = average_precision_score(y_val, val_proba)
precision = precision_score(y_val, val_pred)
recall = recall_score(y_val, val_pred)
f1 = f1_score(y_val, val_pred)
accuracy = accuracy_score(y_val, val_pred)
cm = confusion_matrix(y_val, val_pred)

print("=== LightGBM Baseline — Validation ===")
print(f"PR-AUC   : {pr_auc:.6f}")
print(f"Precision: {precision:.6f}")
print(f"Recall   : {recall:.6f}")
print(f"F1       : {f1:.6f}")
print(f"Accuracy : {accuracy:.6f}")

print("\nConfusion Matrix:")
print(cm)

=== LightGBM Baseline — Validation ===
PR-AUC   : 0.860434
Precision: 0.718478
Recall   : 0.864685
F1       : 0.784830
Accuracy : 0.839184

Confusion Matrix:
[[4603  969]
 [ 387 2473]]


In [8]:
# Tuning split
tune_train_mask = (
    (df["date"] >= "2026-05-08") &
    (df["date"] < "2026-07-01")
)

tune_val_mask = (
    (df["date"] >= "2026-07-01") &
    (df["date"] < "2026-08-01")
)

X_tune_train = X[tune_train_mask]
y_tune_train = y[tune_train_mask]

X_tune_val = X[tune_val_mask]
y_tune_val = y[tune_val_mask]

print("Tuning Train:", X_tune_train.shape)
print("Tuning Validation:", X_tune_val.shape)

print("\nPositive rate:")
print("Train:", y_tune_train.mean())
print("Val  :", y_tune_val.mean())

Tuning Train: (14688, 20)
Tuning Validation: (8432, 20)

Positive rate:
Train: 0.08272058823529412
Val  : 0.15868121442125238


In [9]:
neg_tune = (y_tune_train == 0).sum()
pos_tune = (y_tune_train == 1).sum()

scale_pos_weight_tune = neg_tune / pos_tune

print("Negative:", neg_tune)
print("Positive:", pos_tune)
print("Scale Pos Weight:", scale_pos_weight_tune)

Negative: 13473
Positive: 1215
Scale Pos Weight: 11.088888888888889


In [10]:
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score

configs = {
    "LGBM-1": {
        "n_estimators": 200,
        "learning_rate": 0.05,
        "num_leaves": 15,
        "max_depth": -1,
        "min_child_samples": 20
    },
    
    "LGBM-2": {
        "n_estimators": 300,
        "learning_rate": 0.03,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 20
    },
    
    "LGBM-3": {
        "n_estimators": 400,
        "learning_rate": 0.03,
        "num_leaves": 15,
        "max_depth": -1,
        "min_child_samples": 20
    },
    
    "LGBM-4": {
        "n_estimators": 300,
        "learning_rate": 0.03,
        "num_leaves": 15,
        "max_depth": 5,
        "min_child_samples": 30
    }
}

In [11]:
tuning_results = []

for name, params in configs.items():
    
    model = LGBMClassifier(
        objective="binary",
        scale_pos_weight=scale_pos_weight_tune,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
        **params
    )
    
    model.fit(
        X_tune_train,
        y_tune_train
    )
    
    val_proba = model.predict_proba(X_tune_val)[:, 1]
    
    pr_auc = average_precision_score(
        y_tune_val,
        val_proba
    )
    
    tuning_results.append({
        "Model": name,
        "PR-AUC": pr_auc,
        **params
    })

tuning_results_df = pd.DataFrame(tuning_results)

tuning_results_df = tuning_results_df.sort_values(
    "PR-AUC",
    ascending=False
)

display(tuning_results_df)

,Model,PR-AUC,n_estimators,learning_rate,num_leaves,max_depth,min_child_samples
3,LGBM-4,0.581936,300,0.03,15,5,30
1,LGBM-2,0.573590,300,0.03,31,-1,20
0,LGBM-1,0.570570,200,0.05,15,-1,20
2,LGBM-3,0.568090,400,0.03,15,-1,20


In [12]:
best_lgbm_config_name = tuning_results_df.iloc[0]["Model"]

best_params = configs[best_lgbm_config_name]

print("Best configuration:", best_lgbm_config_name)
print(best_params)

Best configuration: LGBM-4
{'n_estimators': 300, 'learning_rate': 0.03, 'num_leaves': 15, 'max_depth': 5, 'min_child_samples': 30}


In [13]:
best_lgbm = LGBMClassifier(
    objective="binary",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
    **best_params
)

best_lgbm.fit(
    X_train,
    y_train
)

print("Final LightGBM training selesai.")

Final LightGBM training selesai.


In [14]:
val_proba_lgbm = best_lgbm.predict_proba(X_val)[:, 1]

val_pred_lgbm = (
    val_proba_lgbm >= 0.5
).astype(int)

print("=== Tuned LightGBM — Validation ===")

print(
    f"PR-AUC   : "
    f"{average_precision_score(y_val, val_proba_lgbm):.6f}"
)

print(
    f"Precision: "
    f"{precision_score(y_val, val_pred_lgbm):.6f}"
)

print(
    f"Recall   : "
    f"{recall_score(y_val, val_pred_lgbm):.6f}"
)

print(
    f"F1       : "
    f"{f1_score(y_val, val_pred_lgbm):.6f}"
)

print(
    f"Accuracy : "
    f"{accuracy_score(y_val, val_pred_lgbm):.6f}"
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred_lgbm))

=== Tuned LightGBM — Validation ===
PR-AUC   : 0.887945
Precision: 0.675173
Recall   : 0.952797
F1       : 0.790313
Accuracy : 0.828510

Confusion Matrix:
[[4261 1311]
 [ 135 2725]]
